# 03 - Detectability of an excitation in the 4D-STEM signal

How much does an excitation of the object change what the camera actually measures, and
at what dose does that change become measurable rather than merely nonzero?

The object carries an excitation as a real phase perturbation,

$$O_1 = O_0\,e^{ih},\qquad h\ \text{real},$$

and to first order the change in detector intensity at scan position $R$ is the local
intensity Jacobian applied to it,

$$\Delta I_R \;=\; A_R h \;=\; -2\,\mathrm{Im}\!\left[\overline{\Psi_{0,R}}\;
\mathrm{crop}\,\mathcal{F}\{\psi_{0,R}\,h\}\right],
\qquad \psi_{0,R}=P_R O_0,\quad \Psi_{0,R}=\mathrm{crop}\,\mathcal{F}\{\psi_{0,R}\}.$$

This notebook is about how to **score** $\Delta I$, not how to compute it - that is
`linearized_jacobian.jacobian_matvec`.

## Why $\|\Delta I\|_2$ is the wrong metric

A plain L2 norm carries no noise model, so it can only say that a change is nonzero -
never that it is measurable. It also weights every detector pixel equally, which is
backwards. A change of $10^{-6}$ on a bright-field pixel carrying $10^{-2}$ of the
intensity is buried in shot noise; the same change on a pixel carrying $10^{-6}$ is a
doubling of that pixel. L2 scores those two identically.

It is also blind to dose, which is the one experimental knob that decides the answer.

## The right metric: the Poisson-weighted chi-square

The camera counts electrons, so the noise is Poisson and the variance of a pixel **is**
its mean. Weighting each pixel by its own variance gives the deflection signal-to-noise
ratio of the optimal (likelihood-ratio) test:

$$\boxed{\;d^{2}\;=\;\sum_{R,k}\frac{\bigl(\Delta I_{R,k}\bigr)^{2}}{I_{R,k}}\;}
\qquad\text{in \textbf{counts}, summed over scan positions }R\text{ and detector pixels }k.$$

Same algebraic form as L2, but each pixel divided by its own variance.

Three facts make this the correct metric rather than a merely plausible one.

**It is the Fisher information for the excitation amplitude.** Parametrize the measured
intensity as $I(a) = I_0 + a\,A_R h$, where $a=1$ is the excitation as modelled. The
Fisher information for $a$ under the Poisson model is exactly $\mathcal{F}(a) = d^2$, so
the Cramer-Rao bound gives

$$\sigma_a \;=\; 1/d .$$

No estimator, and so no reconstruction algorithm written later, beats that. This is why
the metric is worth computing before building any inversion: it bounds what the
experiment can possibly yield.

**It is the divergence that governs error probability.** To second order in $\Delta I$,
$d^2$ equals the symmetrized Kullback-Leibler divergence between the two Poisson models,
which is the exponent controlling how the probability of confusing them falls with dose.

**It tells you how to build the estimator, not just how to score it.** The optimal test
statistic implied by it is a matched filter over the recorded counts $n_k$,

$$T \;=\; \sum_{R,k} w_{R,k}\,n_{R,k},\qquad w_{R,k}\;\propto\;\frac{\Delta I_{R,k}}{I_{R,k}},$$

whose deflection SNR is $d$.

Report $\sigma_a = 1/d$ rather than $d$ itself. "The excitation amplitude is measured to
$\pm 16\%$" is a statement that can be acted on; a bare $d$ needs a threshold convention
attached before it means anything.

## Three things to get right

### Units: the formula only works in counts

abTEM returns intensities normalized so that $\sum_k I_{0,k} = 1$ at each scan position -
a probability over detector pixels, not a count. With $N$ electrons incident per scan
position the expected counts are $N I_{0,k}$ and the perturbation is $N\,\Delta I_k$, so

$$d^2 \;=\; N \sum_{R,k}\frac{(\Delta I_{R,k})^2}{I_{R,k}}
\qquad\text{with }I,\Delta I\text{ in the normalized units.}$$

Multiply by electrons-per-position **before** summing, or dose silently drops out of the
answer and $d$ becomes meaningless. Detectability scales as $\sqrt{\text{dose}}$.

### The low-intensity floor

$(\Delta I)^2 / I_0$ formally diverges as $I_0 \to 0$, so the dark field looks infinitely
informative. Two reasons it usually is not, and one reason to stay careful:

* The linear response vanishes there as well. Outside the bright-field disc
  $\Psi_{0}=0$, so $A_R h \equiv 0$ and the ratio is $0/0$ rather than divergent.
* Those pixels receive almost no electrons, so in a real exposure they are recorded as
  zero and carry little.
* **But** the true change in the dark field is the *quadratic* term
  $|\mathcal{F}\{\psi_0 h\}|^2$, which the linearization discards. So $d^2$ computed on
  the linear model is a lower bound there.

Do not pick a floor and quote one number. Sweep the pixel cut and report how $d^2$ moves
with it; a metric that is stable across several decades of cut is trustworthy, one that
is not is being driven by near-empty pixels.

### Nuisance parameters

$d^2$ as written answers *"can I distinguish these two known intensity patterns"*. The
experimental question is *"can I detect the excitation when the defocus, the scan
positions, the dose normalization and the base object are not known exactly"*. Any
component of $\Delta I$ that an error in one of those nuisance parameters could imitate
is not evidence of the excitation.

The honest number projects those directions out in the same Fisher metric,

$$d^2_{\text{eff}} \;=\;
\bigl\|\,(1 - P_{\text{nuisance}})\,W^{1/2} A_R h\,\bigr\|^{2},
\qquad W = \mathrm{diag}\!\left(1/I_0\right),$$

where the columns of the nuisance basis are $\partial I_0/\partial\theta$ for each
unknown $\theta$ - obtained by finite differences of the base intensity - orthonormalized
in the $W$ inner product before projecting.

This is not a small correction. The leading modes of $A_R$ are the translation dipole
pair, so an excitation with any dipole component competes directly with scan-position
jitter.

## Setup

Configuration lives in the single cell below; nothing further down should need editing
per experiment.

In [ ]:
import sys
import pickle
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import linearized_jacobian as lj

print("notebook dir :", NOTEBOOK_DIR)
print("module       :", lj.__file__)

In [ ]:
PHASE_CACHE_PATH = NOTEBOOK_DIR.parent / "bcenters_base_excited_phases_wide_fov.pkl"
PHASE_CACHE_KEY_BASE = "base"
PHASE_CACHE_KEY_EXCITED = "excited"

ACC_VOLTAGE_EV = 60e3
SLICE_THICKNESS_ANG = 1.0
PROBE_SEMIANGLE_MRAD = 30.0
PROBE_DEFOCUS_ANG = 0.0
SCAN_STEP_SIZE_ANG = 1.5
SCAN_ENDPOINT = True

# Pixel cuts to sweep when reporting d^2, as a fraction of the peak base intensity.
INTENSITY_FLOOR_FRACTIONS = [1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 0.0]

# Doses to tabulate, in electrons per square Angstrom over the scanned area.
DOSE_SWEEP_E_PER_ANG2 = [1e3, 1e4, 1e5, 1e6]

print("semiangle [mrad]      :", PROBE_SEMIANGLE_MRAD)
print("defocus [Angstrom]    :", PROBE_DEFOCUS_ANG)
print("scan step [Angstrom]  :", SCAN_STEP_SIZE_ANG)
print("intensity floors      :", INTENSITY_FLOOR_FRACTIONS)
print("doses [e-/A^2]        :", DOSE_SWEEP_E_PER_ANG2)

In [ ]:
with open(PHASE_CACHE_PATH, "rb") as handle:
    phase_cache = pickle.load(handle)

phase_base = np.asarray(phase_cache[PHASE_CACHE_KEY_BASE], dtype=np.float32)
phase_excited = np.asarray(phase_cache[PHASE_CACHE_KEY_EXCITED], dtype=np.float32)
object_x_ang = np.asarray(phase_cache["x"], dtype=np.float32)
object_y_ang = np.asarray(phase_cache["y"], dtype=np.float32)
object_sampling_ang = (
    float(object_y_ang[1] - object_y_ang[0]),
    float(object_x_ang[1] - object_x_ang[0]),
)

# The excitation is the raw phase difference. No apodization: the Jacobian must be fed
# the same perturbation that produced the data it is being compared against.
excitation_phase_rad = (phase_excited - phase_base).astype(np.float32)
object_base = np.exp(1j * phase_base).astype(np.complex64)

potential = lj.make_phase_potential(
    phase_base, ACC_VOLTAGE_EV, SLICE_THICKNESS_ANG, object_sampling_ang
)
probe = lj.make_probe(
    ACC_VOLTAGE_EV, PROBE_SEMIANGLE_MRAD, PROBE_DEFOCUS_ANG, potential
)
detector_shape = lj.valid_detector_shape(probe)
scan_positions_ang = lj.grid_scan_positions_ang(
    potential, SCAN_STEP_SIZE_ANG, endpoint=SCAN_ENDPOINT
)
scan_area_ang2 = float(potential.extent[0]) * float(potential.extent[1])

print("object grid              :", phase_base.shape)
print("object sampling [A/px]   :", object_sampling_ang)
print("scan positions           :", scan_positions_ang.shape[:2],
      "=", int(np.prod(scan_positions_ang.shape[:2])))
# abTEM returns (n_0, n_1, 2) with the last axis (x, y). Which array axis steps which
# coordinate is a recurring trap, so read both steps back rather than assuming.
scan_step_axis0_ang = scan_positions_ang[1, 0] - scan_positions_ang[0, 0]
scan_step_axis1_ang = scan_positions_ang[0, 1] - scan_positions_ang[0, 0]
print("step along array axis 0  : (dx, dy) =", tuple(float(v) for v in scan_step_axis0_ang))
print("step along array axis 1  : (dx, dy) =", tuple(float(v) for v in scan_step_axis1_ang))
print("requested step [A]       :", SCAN_STEP_SIZE_ANG,
      "-- GridScan rounds this to fit the extent, so never assume it")
print("detector shape (valid)   :", detector_shape)
print("scanned area [A^2]       :", scan_area_ang2)
print()
print("excitation max |dphi| [rad] :", float(np.abs(excitation_phase_rad).max()))
print("excitation L2 norm [rad]    :", float(np.linalg.norm(excitation_phase_rad)))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.9), constrained_layout=True)
object_extent_ang = [
    float(object_x_ang[0]), float(object_x_ang[-1]),
    float(object_y_ang[0]), float(object_y_ang[-1]),
]
excitation_vmax = float(np.abs(excitation_phase_rad).max())
for ax, image, title, cmap, limits in [
    (axes[0], phase_base, "base object phase", "Blues", None),
    (axes[1], excitation_phase_rad, "excitation (excited - base)", "RdBu_r",
     excitation_vmax),
]:
    handle = ax.imshow(
        image, extent=object_extent_ang, origin="lower", cmap=cmap,
        interpolation="nearest",
        vmin=(-limits if limits is not None else None),
        vmax=(limits if limits is not None else None),
    )
    ax.set_xlabel("x [$\\AA$]")
    ax.set_ylabel("y [$\\AA$]")
    ax.set_title(title, fontsize=10)
    fig.colorbar(handle, ax=ax, shrink=0.85, label="phase [rad]")
fig.suptitle(
    f"{PHASE_CACHE_PATH.name}   |   {ACC_VOLTAGE_EV/1e3:.0f} kV   |   "
    f"{object_sampling_ang[0]:.3f} $\\AA$/px", fontsize=10,
)
plt.show()

## Not implemented yet

The setup above stops short of the measurement. What follows, in order:

1. **$I_0$ and $\Delta I$ over the scan** - one `jacobian_matvec` per scan position
   against `excitation_phase_rad`, giving the base intensity and the linear response.
2. **$d^2$ with the floor sweep** - the boxed sum over `INTENSITY_FLOOR_FRACTIONS`,
   reported as a table so the stability of the cut is visible rather than assumed.
3. **Dose table** - $d$ and $\sigma_a = 1/d$ at each dose in `DOSE_SWEEP_E_PER_ANG2`,
   and the dose required for a chosen number of sigma.
4. **Nuisance projection** - finite-difference columns $\partial I_0/\partial\theta$ for
   scan position, defocus and dose normalization, orthonormalized in the $W$ inner
   product, and $d^2_{\text{eff}}$ beside the raw $d^2$.
5. **Probe-parameter dependence** - steps 1-4 over a defocus x semiangle grid, which is
   the question "which probe setting best detects *this* excitation". Unlike the generic
   Jacobian modes of notebook 01, the excitation enters by construction.